# Ensemble Classifiers: Kombinace modelů pro lepší predikce

## Co jsou Ensemble Classifiers?

Ensemble Classifiers (Ansámblové klasifikátory) jsou pokročilou technikou strojového učení, která kombinuje predikce z více modelů s cílem dosáhnout lepších výsledků, než by poskytl jakýkoliv samostatný model. Základní myšlenkou je, že "více hlav je moudřejších než jedna" - kombinací různých modelů můžeme eliminovat jejich jednotlivé slabiny a posílit jejich silné stránky.

### Hlavní typy ansámblových metod:

1. **Bagging (Bootstrap Aggregating):** Trénuje několik instancí stejného algoritmu na různých podmnožinách trénovacích dat a kombinuje jejich výstupy. Příkladem je Random Forest.

2. **Boosting:** Postupně staví slabé modely, kde každý nový model se zaměřuje na opravování chyb předchozího. Příklady zahrnují AdaBoost a Gradient Boosting.

3. **Stacking:** Kombinuje predikce z různých typů modelů použitím dalšího modelu (meta-learner), který se učí, jak nejlépe zkombinovat jejich výstupy.

4. **Voting:** Jednoduchá kombinace predikcí více modelů buď prostým hlasováním (hard voting), nebo průměrováním pravděpodobností (soft voting).

### Kdy použít Ensemble Classifiers:

- Když potřebujete maximalizovat výkon predikce a přesnost je klíčová
- Pro komplexní problémy, kde jednotlivé modely nevykazují dostatečnou přesnost
- Když chcete robustní model méně náchylný k přeučení
- Pro soutěže v oblasti strojového učení (často dominují ansámblové metody)
- Když máte dostatek výpočetních zdrojů (některé ansámbly jsou výpočetně náročné)

### Výhody:

- Vyšší přesnost a robustnost ve srovnání s jednotlivými modely
- Snížení variance a rizika přeučení
- Lepší stabilita při šumu v datech
- Možnost zachytit komplexnější vztahy v datech
- Poskytují odhad nejistoty predikce

### Nevýhody:

- Vyšší výpočetní náročnost a čas trénování
- Složitější interpretace ve srovnání s jednoduchými modely
- Potenciálně složitější implementace a ladění
- Mohou vyžadovat více paměti pro uchování více modelů

Pojďme nyní prozkoumat a implementovat různé typy Ensemble klasifikátorů pomocí knihovny scikit-learn.

In [ ]:
# Import základních knihoven
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from time import time
from IPython.display import display

# Potřebné nástroje pro zpracování dat
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score, KFold, learning_curve
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix, 
    roc_curve, roc_auc_score, f1_score, precision_recall_curve, auc
)
from sklearn.inspection import permutation_importance

# Načtení datasetů
from sklearn.datasets import load_breast_cancer, fetch_openml

# Ensemble klasifikátory
from sklearn.ensemble import (
    RandomForestClassifier, GradientBoostingClassifier, 
    AdaBoostClassifier, VotingClassifier, StackingClassifier,
    BaggingClassifier, ExtraTreesClassifier
)

# Základní klasifikátory pro porovnání a kombinaci
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

# Nastavení pro reprodukovatelnost
import warnings
warnings.filterwarnings("ignore")
np.random.seed(42)

# Nastavení vizualizací
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('viridis')
plt.rcParams['figure.figsize'] = (12, 8)

## 1. Načtení a příprava dat

Pro naši první analýzu použijeme dataset rakoviny prsu Wisconsin, který je často používaný pro porovnávání klasifikátorů. Data obsahují různé vlastnosti buněčných jader získaných z digitálních snímků prsní tkáně, a cílem je klasifikovat nádory jako maligní (zhoubné) nebo benigní (nezhoubné).

In [ ]:
# Načtení datasetu rakoviny prsu
cancer = load_breast_cancer()
X = cancer.data
y = cancer.target

# Základní informace o datasetu
print(f"Tvar datasetu: {X.shape}")
print(f"Počet tříd: {len(np.unique(y))}, Názvy tříd: {cancer.target_names}")
print(f"Rozložení tříd: {np.bincount(y)}")
print(f"Názvy příznaků: {cancer.feature_names}")

# Vytvoření DataFrame pro snadnější manipulaci s daty
df = pd.DataFrame(X, columns=cancer.feature_names)
df['target'] = y
df['diagnosis'] = [cancer.target_names[i] for i in y]

# Zobrazení prvních pár řádků dat
display(df.head())

### Průzkumná analýza dat (EDA)

Podívejme se na základní charakteristiku dat a jejich strukturu.

In [ ]:
# Statistický přehled dat
print("Statistický přehled numerických proměnných:")
display(df.describe())

# Podívejme se na korelace mezi příznaky
plt.figure(figsize=(14, 12))
correlation_matrix = df.iloc[:, :-2].corr()
mask = np.triu(np.ones_like(correlation_matrix, dtype=bool))
sns.heatmap(correlation_matrix, mask=mask, cmap='viridis', vmax=1, vmin=-1, 
            center=0, square=True, linewidths=.5, annot=False, fmt='.2f')
plt.title('Korelační matice mezi příznaky', fontsize=16)
plt.tight_layout()
plt.show()

# Vizualizace distribuce vybraných příznaků podle diagnózy
important_features = ['mean radius', 'mean texture', 'mean perimeter', 'mean area', 'mean concavity']

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

for i, feature in enumerate(important_features):
    if i < 6:  # Pouze pro prvních 6 grafů
        sns.boxplot(x='diagnosis', y=feature, data=df, ax=axes[i], palette='viridis')
        axes[i].set_title(f'Distribuce podle diagnózy: {feature}', fontsize=14)
        axes[i].set_xlabel('Diagnóza')
        axes[i].set_ylabel(feature)

plt.tight_layout()
plt.show()

### Příprava dat pro modelování

Rozdělíme data na trénovací a testovací množinu.

In [ ]:
# Rozdělení dat na trénovací a testovací množiny
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

print(f"Trénovací data: {X_train.shape}")
print(f"Testovací data: {X_test.shape}")
print(f"Rozložení tříd v trénovacích datech: {np.bincount(y_train)}")
print(f"Rozložení tříd v testovacích datech: {np.bincount(y_test)}")

## 2. Implementace základních klasifikátorů

Pro srovnání nejprve implementujeme několik jednoduchých klasifikátorů, které následně použijeme pro porovnání s ansámblovými metodami.

In [ ]:
# Definice základních klasifikátorů
base_classifiers = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'SVM': SVC(kernel='rbf', probability=True, random_state=42),
    'KNN': KNeighborsClassifier(n_neighbors=5)
}

# Funkce pro vyhodnocení modelu
def evaluate_classifier(name, classifier, X_train, X_test, y_train, y_test):
    # Trénování
    start_time = time()
    classifier.fit(X_train, y_train)
    train_time = time() - start_time
    
    # Predikce
    start_time = time()
    y_pred = classifier.predict(X_test)
    predict_time = time() - start_time
    
    # Metriky
    accuracy = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    
    # Výstup
    print(f"\n{name}:")
    print(f"Přesnost: {accuracy:.4f}")
    print(f"F1 skóre: {f1:.4f}")
    print(f"Čas trénování: {train_time:.4f} s")
    print(f"Čas predikce: {predict_time:.4f} s")
    print("\nKlasifikační report:")
    print(classification_report(y_test, y_pred, target_names=cancer.target_names))
    
    # Matice záměn
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False,
                xticklabels=cancer.target_names, yticklabels=cancer.target_names)
    plt.title(f'Matice záměn - {name}', fontsize=14)
    plt.ylabel('Skutečná třída')
    plt.xlabel('Predikovaná třída')
    plt.tight_layout()
    plt.show()
    
    return {
        'name': name,
        'accuracy': accuracy,
        'f1': f1,
        'train_time': train_time,
        'predict_time': predict_time,
        'model': classifier
    }

# Standardizace dat pro lepší výkon některých algoritmů
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Vyhodnocení základních klasifikátorů
base_results = []
for name, clf in base_classifiers.items():
    result = evaluate_classifier(name, clf, X_train_scaled, X_test_scaled, y_train, y_test)
    base_results.append(result)

## 3. Random Forest

Random Forest je metoda bagging, která vytváří soubor rozhodovacích stromů a kombinuje jejich rozhodnutí. Každý strom je trénován na náhodně vybrané podmnožině dat a příznaků.

In [ ]:
# Implementace Random Forest klasifikátoru
rf_classifier = RandomForestClassifier(
    n_estimators=100,  # počet stromů v lese
    max_features='sqrt',  # počet příznaků k považování při každém větvení
    max_depth=None,  # maximální hloubka stromů (None = neomezená)
    min_samples_split=2,  # minimální počet vzorků potřebných pro rozdělení
    bootstrap=True,  # použití bootstrappingu (výběru s opakováním)
    random_state=42
)

# Vyhodnocení Random Forest
rf_result = evaluate_classifier(
    "Random Forest", rf_classifier, X_train_scaled, X_test_scaled, y_train, y_test
)
base_results.append(rf_result)

### Vizualizace důležitosti příznaků v Random Forest

Jednou z výhod Random Forest je možnost zjistit důležitost jednotlivých příznaků pro klasifikaci.

In [ ]:
# Získání důležitosti příznaků
feature_importance = rf_classifier.feature_importances_

# Vytvoření DataFrame pro vizualizaci
importance_df = pd.DataFrame({
    'Feature': cancer.feature_names,
    'Importance': feature_importance
})
importance_df = importance_df.sort_values('Importance', ascending=False)

# Vizualizace top 15 nejdůležitějších příznaků
plt.figure(figsize=(12, 8))
sns.barplot(x='Importance', y='Feature', data=importance_df.head(15), palette='viridis')
plt.title('Top 15 nejdůležitějších příznaků podle Random Forest', fontsize=16)
plt.tight_layout()
plt.show()

## 4. Gradient Boosting

Gradient Boosting je postupná metoda, která buduje modely sekvenčně, přičemž každý nový model se snaží opravit chyby předchozích modelů.

In [ ]:
# Implementace Gradient Boosting klasifikátoru
gb_classifier = GradientBoostingClassifier(
    n_estimators=100,  # počet slabých modelů (stromů)
    learning_rate=0.1,  # určuje, jak rychle se model učí
    max_depth=3,  # maximální hloubka jednotlivých stromů
    subsample=1.0,  # podíl vzorků použitých pro trénování jednotlivých stromů
    random_state=42
)

# Vyhodnocení Gradient Boosting
gb_result = evaluate_classifier(
    "Gradient Boosting", gb_classifier, X_train_scaled, X_test_scaled, y_train, y_test
)
base_results.append(gb_result)

### Analýza vývoje chyby v průběhu přidávání slabých modelů

In [ ]:
# Získání chyby trénování a testování v průběhu iterací
gb_classifier_analysis = GradientBoostingClassifier(
    n_estimators=500,  # zvýšíme počet pro lepší vizualizaci trendu
    learning_rate=0.1,
    max_depth=3,
    subsample=1.0,
    random_state=42
)
gb_classifier_analysis.fit(X_train_scaled, y_train)

# Extrakce chyb
train_errors = np.zeros(500)
for i, pred in enumerate(gb_classifier_analysis.staged_predict(X_train_scaled)):
    train_errors[i] = 1.0 - accuracy_score(y_train, pred)
    
test_errors = np.zeros(500)
for i, pred in enumerate(gb_classifier_analysis.staged_predict(X_test_scaled)):
    test_errors[i] = 1.0 - accuracy_score(y_test, pred)
    
# Vizualizace vývoje chyby
plt.figure(figsize=(10, 6))
plt.plot(np.arange(1, 501), train_errors, label='Chyba na trénovacích datech')
plt.plot(np.arange(1, 501), test_errors, label='Chyba na testovacích datech')
plt.axvline(x=100, color='red', linestyle='--', label='Výchozí počet estimátorů')
plt.xlabel('Počet iterací (stromů)', fontsize=12)
plt.ylabel('Chyba (1 - přesnost)', fontsize=12)
plt.title('Vývoj chyby Gradient Boosting klasifikátoru', fontsize=16)
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

## 5. AdaBoost

AdaBoost (Adaptive Boosting) je další boosting metoda, která iterativně přizpůsobuje váhy trénovacích vzorků na základě chyb předchozích modelů.

In [ ]:
# Implementace AdaBoost klasifikátoru
ada_classifier = AdaBoostClassifier(
    base_estimator=DecisionTreeClassifier(max_depth=1),  # slabý base model - pařez (stump)
    n_estimators=100,  # počet slabých modelů
    learning_rate=1.0,  # rychlost učení
    algorithm='SAMME.R',  # algoritmus použitý pro AdaBoost
    random_state=42
)

# Vyhodnocení AdaBoost
ada_result = evaluate_classifier(
    "AdaBoost", ada_classifier, X_train_scaled, X_test_scaled, y_train, y_test
)
base_results.append(ada_result)

## 6. Extra Trees (Extremely Randomized Trees)

Extra Trees je varianta Random Forest, která přidává další náhodnost při budování jednotlivých stromů.

In [ ]:
# Implementace Extra Trees klasifikátoru
et_classifier = ExtraTreesClassifier(
    n_estimators=100,
    max_features='sqrt',
    bootstrap=False,  # na rozdíl od Random Forest, výchozí hodnota je False
    random_state=42
)

# Vyhodnocení Extra Trees
et_result = evaluate_classifier(
    "Extra Trees", et_classifier, X_train_scaled, X_test_scaled, y_train, y_test
)
base_results.append(et_result)

## 7. Voting Classifier

Voting Classifier kombinuje několik různých modelů a rozhoduje na základě hlasování (buď tvrdého nebo měkkého).

In [ ]:
# Vytvoření jednotlivých klasifikátorů pro Voting
voting_classifiers = [
    ('rf', RandomForestClassifier(n_estimators=100, random_state=42)),
    ('svm', SVC(kernel='rbf', probability=True, random_state=42)),
    ('lr', LogisticRegression(max_iter=1000, random_state=42)),
    ('gb', GradientBoostingClassifier(n_estimators=100, random_state=42))
]

# Implementace Hard Voting (rozhodování na základě většiny hlasů)
hard_voting = VotingClassifier(
    estimators=voting_classifiers,
    voting='hard'
)

# Implementace Soft Voting (rozhodování na základě průměrných pravděpodobností)
soft_voting = VotingClassifier(
    estimators=voting_classifiers,
    voting='soft'
)

# Vyhodnocení Hard Voting
hard_voting_result = evaluate_classifier(
    "Hard Voting", hard_voting, X_train_scaled, X_test_scaled, y_train, y_test
)
base_results.append(hard_voting_result)

# Vyhodnocení Soft Voting
soft_voting_result = evaluate_classifier(
    "Soft Voting", soft_voting, X_train_scaled, X_test_scaled, y_train, y_test
)
base_results.append(soft_voting_result)

## 8. Stacking Classifier

Stacking kombinuje několik základních modelů pomocí meta-modelu, který se učí, jak optimálně kombinovat jejich predikce.

In [ ]:
# Definice základních modelů pro stacking
base_models = [
    ('rf', RandomForestClassifier(n_estimators=100, random_state=42)),
    ('et', ExtraTreesClassifier(n_estimators=100, random_state=42)),
    ('gb', GradientBoostingClassifier(n_estimators=100, random_state=42)),
    ('svm', SVC(kernel='rbf', probability=True, random_state=42))
]

# Implementace Stacking Classifier s logistickou regresí jako meta-klasifikátorem
stacking_classifier = StackingClassifier(
    estimators=base_models,
    final_estimator=LogisticRegression(max_iter=1000, random_state=42),
    cv=5,  # počet foldů pro Cross-Validation
    stack_method='predict_proba',  # použít pravděpodobnosti
    n_jobs=-1  # použít všechna dostupná CPU jádra
)

# Vyhodnocení Stacking
stacking_result = evaluate_classifier(
    "Stacking Classifier", stacking_classifier, X_train_scaled, X_test_scaled, y_train, y_test
)
base_results.append(stacking_result)

## 9. Bagging Classifier

Bagging (Bootstrap Aggregating) vytváří více instancí stejného modelu trénovaných na různých podmnožinách dat vybraných s opakováním.

In [ ]:
# Implementace Bagging s rozhodovacím stromem jako základním modelem
bagging_classifier = BaggingClassifier(
    base_estimator=DecisionTreeClassifier(),  # základní model
    n_estimators=100,  # počet modelů
    max_samples=0.8,  # podíl vzorků pro každý model
    max_features=0.8,  # podíl příznaků pro každý model
    bootstrap=True,  # vzorkování s opakováním
    bootstrap_features=False,  # bez vzorkování příznaků
    random_state=42
)

# Vyhodnocení Bagging
bagging_result = evaluate_classifier(
    "Bagging Classifier", bagging_classifier, X_train_scaled, X_test_scaled, y_train, y_test
)
base_results.append(bagging_result)

## 10. Srovnání všech klasifikátorů

Nyní porovnáme výkon všech implementovaných klasifikátorů.

In [ ]:
# Vytvoření DataFrame pro srovnání
comparison_df = pd.DataFrame(base_results)
comparison_df = comparison_df[['name', 'accuracy', 'f1', 'train_time', 'predict_time']]
comparison_df = comparison_df.sort_values('accuracy', ascending=False)

# Zobrazení srovnání
print("Srovnání klasifikátorů:")
display(comparison_df)

# Vizualizace srovnání přesnosti
plt.figure(figsize=(14, 8))

# Přesnost
plt.subplot(1, 2, 1)
sns.barplot(x='name', y='accuracy', data=comparison_df, palette='viridis')
plt.title('Přesnost jednotlivých klasifikátorů', fontsize=14)
plt.xticks(rotation=45, ha='right')
plt.xlabel('Klasifikátor')
plt.ylabel('Přesnost')
plt.ylim(0.85, 1.0)  # Upravíme škálu pro lepší viditelnost rozdílů

# Čas trénování
plt.subplot(1, 2, 2)
sns.barplot(x='name', y='train_time', data=comparison_df, palette='viridis')
plt.title('Čas trénování jednotlivých klasifikátorů', fontsize=14)
plt.xticks(rotation=45, ha='right')
plt.xlabel('Klasifikátor')
plt.ylabel('Čas [s]')

plt.tight_layout()
plt.show()

## 11. Hyperparameter tuning s GridSearchCV pro nejlepší modely

Vybereme dva nejlepší modely a optimalizujeme jejich hyperparametry pomocí GridSearchCV.

In [ ]:
# Získání nejlepších dvou modelů na základě přesnosti
best_models = comparison_df.head(2)['name'].values
print(f"Optimalizujeme hyperparametry pro: {best_models}")

# Funkce pro optimalizaci modelu
def optimize_model(model_name, param_grid):
    print(f"\nOptimalizace hyperparametrů pro {model_name}...")
    
    if model_name == "Random Forest":
        model = RandomForestClassifier(random_state=42)
    elif model_name == "Gradient Boosting":
        model = GradientBoostingClassifier(random_state=42)
    elif model_name == "Extra Trees":
        model = ExtraTreesClassifier(random_state=42)
    elif model_name == "Stacking Classifier":
        # Pro Stacking je optimalizace složitější, vrátíme výchozí model
        return stacking_classifier
    elif model_name == "Soft Voting":
        # Pro Voting je také optimalizace složitější, vrátíme výchozí model
        return soft_voting
    else:
        print(f"Model {model_name} není definován pro optimalizaci.")
        return None
    
    # Grid Search
    grid_search = GridSearchCV(
        model, param_grid, cv=5, scoring='accuracy', n_jobs=-1, verbose=1
    )
    
    # Trénování
    start_time = time()
    grid_search.fit(X_train_scaled, y_train)
    search_time = time() - start_time
    
    # Výsledky
    print(f"Nejlepší skóre: {grid_search.best_score_:.4f}")
    print(f"Nejlepší parametry: {grid_search.best_params_}")
    print(f"Čas hledání: {search_time:.2f} s")
    
    return grid_search.best_estimator_

# Definice prostorů parametrů pro jednotlivé modely
param_grids = {}

# Random Forest parametry
param_grids["Random Forest"] = {
    'n_estimators': [50, 100, 200],
    'max_features': ['sqrt', 'log2', None],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

# Gradient Boosting parametry
param_grids["Gradient Boosting"] = {
    'n_estimators': [50, 100, 200],
    'learning_rate': [0.01, 0.1, 0.2],
    'max_depth': [3, 4, 5],
    'subsample': [0.8, 1.0],
    'min_samples_split': [2, 5]
}

# Extra Trees parametry
param_grids["Extra Trees"] = {
    'n_estimators': [50, 100, 200],
    'max_features': ['sqrt', 'log2', None],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5, 10]
}

# Optimalizace nejlepších dvou modelů
optimized_models = {}
for model_name in best_models:
    if model_name in param_grids:
        optimized_models[model_name] = optimize_model(model_name, param_grids[model_name])
    else:
        print(f"Pro model {model_name} nejsou definovány parametry pro optimalizaci.")

### Vyhodnocení optimalizovaných modelů

In [ ]:
# Vyhodnocení optimalizovaných modelů
optimized_results = []

for name, model in optimized_models.items():
    result = evaluate_classifier(
        f"Optimalizovaný {name}", model, X_train_scaled, X_test_scaled, y_train, y_test
    )
    optimized_results.append(result)

## 12. Křivky učení optimalizovaného modelu

Podíváme se na křivky učení pro nejlepší optimalizovaný model, abychom zhodnotili jeho výkon při různých velikostech trénovacího datasetu.

In [ ]:
# Výběr nejlepšího modelu z optimalizovaných
best_model_name = best_models[0]
best_model = optimized_models.get(best_model_name)

if best_model is None:
    # Pokud nebyla definována optimalizace, vyberte model přímo
    if best_model_name == "Stacking Classifier":
        best_model = stacking_classifier
    elif best_model_name == "Soft Voting":
        best_model = soft_voting

# Generování křivek učení
train_sizes, train_scores, test_scores = learning_curve(
    best_model, X_scaled, y, 
    train_sizes=np.linspace(0.1, 1.0, 10), 
    cv=5, 
    scoring='accuracy', 
    n_jobs=-1
)

# Výpočet průměrů a směrodatných odchylek
train_mean = np.mean(train_scores, axis=1)
train_std = np.std(train_scores, axis=1)
test_mean = np.mean(test_scores, axis=1)
test_std = np.std(test_scores, axis=1)

# Vizualizace křivek učení
plt.figure(figsize=(10, 6))
plt.plot(train_sizes, train_mean, 'o-', color='#2A9D8F', label='Trénovací skóre')
plt.plot(train_sizes, test_mean, 'o-', color='#E9C46A', label='Validační skóre')
plt.fill_between(train_sizes, train_mean - train_std, train_mean + train_std, alpha=0.1, color='#2A9D8F')
plt.fill_between(train_sizes, test_mean - test_std, test_mean + test_std, alpha=0.1, color='#E9C46A')
plt.xlabel('Počet trénovacích vzorků', fontsize=12)
plt.ylabel('Přesnost', fontsize=12)
plt.title(f'Křivky učení pro {best_model_name}', fontsize=16)
plt.legend(loc='lower right', fontsize=12)
plt.grid(True)
plt.ylim(0.8, 1.01)
plt.tight_layout()
plt.show()

## 13. Analýza ROC křivek

Pro hodnocení binárních klasifikátorů je užitečné prozkoumat ROC křivky, které ukazují kompromis mezi senzitivitou a specificitou.

In [ ]:
# Funkce pro vykreslení ROC křivky
def plot_roc_curve(models_dict):
    plt.figure(figsize=(10, 8))
    
    for name, model in models_dict.items():
        # Získání pravděpodobností pro pozitivní třídu
        if hasattr(model, "predict_proba"):
            try:
                y_score = model.predict_proba(X_test_scaled)[:, 1]
                fpr, tpr, _ = roc_curve(y_test, y_score)
                roc_auc = auc(fpr, tpr)
                plt.plot(fpr, tpr, lw=2, label=f'{name} (AUC = {roc_auc:.4f})')
            except:
                print(f"Nelze získat pravděpodobnosti pro {name}")
        else:
            print(f"Model {name} nemá metodu predict_proba")
    
    # Vykreslení náhodné baseline
    plt.plot([0, 1], [0, 1], color='gray', linestyle='--', label='Náhodná klasifikace')
    
    # Formátování grafu
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate (1 - Specificita)', fontsize=12)
    plt.ylabel('True Positive Rate (Senzitivita)', fontsize=12)
    plt.title('ROC křivky pro různé klasifikátory', fontsize=16)
    plt.legend(loc="lower right", fontsize=10)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

# Vybrané modely pro srovnání
models_to_compare = {
    'Random Forest': rf_classifier,
    'Gradient Boosting': gb_classifier,
    'Extra Trees': et_classifier,
    'Stacking': stacking_classifier,
    'Logistic Regression': base_classifiers['Logistic Regression']
}

# Přidání optimalizovaných modelů
for name, model in optimized_models.items():
    models_to_compare[f'Optimalizovaný {name}'] = model

# Vykreslení ROC křivek
plot_roc_curve(models_to_compare)

## 14. Testování na jiném datasetu

Pro ověření robustnosti našich modelů je dobré je otestovat na jiném datasetu. Použijeme dataset Coimbra Breast Cancer, který je podobný, ale obsahuje jiné příznaky.

In [ ]:
# Načtení dalšího datasetu - zkusíme Wine dataset
wine = fetch_openml(name='wine', version=1, as_frame=False)
X_wine = wine.data
y_wine = (wine.target.astype(int) == 1).astype(int)  # Převod na binární problém

print(f"Tvar Wine datasetu: {X_wine.shape}")
print(f"Distribuce tříd: {np.bincount(y_wine)}")

# Rozdělení na trénovací a testovací množiny
X_train_wine, X_test_wine, y_train_wine, y_test_wine = train_test_split(
    X_wine, y_wine, test_size=0.25, random_state=42, stratify=y_wine
)

# Standardizace
scaler_wine = StandardScaler()
X_train_wine_scaled = scaler_wine.fit_transform(X_train_wine)
X_test_wine_scaled = scaler_wine.transform(X_test_wine)

# Trénování a vyhodnocení nejlepšího modelu na novém datasetu
best_model_name = best_models[0]
best_model = optimized_models.get(best_model_name)

if best_model is None:
    # Pokud nebyla definována optimalizace, vyberte model přímo
    if best_model_name == "Stacking Classifier":
        best_model = stacking_classifier
    elif best_model_name == "Soft Voting":
        best_model = soft_voting

# Resetování a trénování modelu na novém datasetu
best_model.fit(X_train_wine_scaled, y_train_wine)

# Predikce a vyhodnocení
y_pred_wine = best_model.predict(X_test_wine_scaled)
accuracy_wine = accuracy_score(y_test_wine, y_pred_wine)

print(f"\nVýkon {best_model_name} na Wine datasetu:")
print(f"Přesnost: {accuracy_wine:.4f}")
print("\nKlasifikační report:")
print(classification_report(y_test_wine, y_pred_wine))

# Matice záměn
cm_wine = confusion_matrix(y_test_wine, y_pred_wine)
plt.figure(figsize=(8, 6))
sns.heatmap(cm_wine, annot=True, fmt="d", cmap="Blues", cbar=False,
            xticklabels=['Třída 0', 'Třída 1'], yticklabels=['Třída 0', 'Třída 1'])
plt.title(f'Matice záměn - {best_model_name} na Wine datasetu', fontsize=14)
plt.ylabel('Skutečná třída')
plt.xlabel('Predikovaná třída')
plt.tight_layout()
plt.show()

## 15. Závěr

### Shrnutí výsledků

V tomto notebooku jsme provedli komplexní analýzu a porovnání různých ansámblových klasifikačních metod. Zde jsou hlavní zjištění:

In [ ]:
# Finální srovnání všech modelů včetně optimalizovaných
all_results = base_results + optimized_results
all_comparison = pd.DataFrame(all_results)[['name', 'accuracy', 'f1', 'train_time', 'predict_time']]
all_comparison = all_comparison.sort_values('accuracy', ascending=False)

# Zobrazení finálního srovnání
print("Finální srovnání všech klasifikátorů:")
display(all_comparison.head(10))

# Vizualizace finálního srovnání přesnosti
plt.figure(figsize=(14, 8))
sns.barplot(x='name', y='accuracy', data=all_comparison.head(10), palette='viridis')
plt.title('Top 10 klasifikátorů podle přesnosti', fontsize=16)
plt.xticks(rotation=45, ha='right')
plt.xlabel('Klasifikátor', fontsize=12)
plt.ylabel('Přesnost', fontsize=12)
plt.ylim(0.90, 1.0)  # Upravíme škálu pro lepší viditelnost rozdílů
plt.grid(axis='y')
plt.tight_layout()
plt.show()

### Klíčové poznatky o Ensemble klasifikátorech:

1. **Vynikající výkon**: Ensemble metody jako Random Forest, Gradient Boosting a Stacking konzistentně předčily jednotlivé modely v přesnosti klasifikace, což potvrzuje jejich sílu v kombinování znalostí různých modelů.

2. **Trade-off mezi přesností a výpočetní náročností**: Složitější ensemble techniky jako Stacking a Soft Voting dosahovaly vyšší přesnosti, ale za cenu výrazně vyššího času trénování a predikce.

3. **Optimalizace hyperparametrů**: Ladění hyperparametrů ukázalo být klíčové, jelikož optimalizované modely dosahovaly ještě lepších výsledků než jejich výchozí konfigurace.

4. **Robustnost**: Ensemble metody prokázaly svou robustnost tím, že si udržely vysoký výkon i na jiném datasetu, což naznačuje jejich dobrou generalizační schopnost.

5. **Důležitost příznaků**: Random Forest a jiné tree-based metody nám poskytly cenný pohled na důležitost jednotlivých příznaků, což může být užitečné pro další analýzu a selekci příznaků.

### Doporučení pro praktické použití:

- **Pro rychlý, kvalitní výsledek**: Random Forest je výbornou volbou díky kombinaci vysoké přesnosti, rozumné výpočetní náročnosti a jednoduché interpretaci.

- **Pro maximální přesnost**: Gradient Boosting nebo Stacking jsou nejlepší volbou, pokud je důležitější přesnost než rychlost a interpretovatelnost.

- **Pro velké datasety**: Extra Trees nebo optimalizovaný Random Forest poskytují dobrý kompromis mezi rychlostí a přesností.

- **Pro interpretovatelnost**: Přestože jsou ensemble metody obecně hůře interpretovatelné, Random Forest nabízí užitečné metriky důležitosti příznaků.

- **Pro produkční nasazení**: Zvažte kompromis mezi přesností a výpočetní náročností - někdy může být jednodušší model s mírně nižší přesností lepší volbou z hlediska efektivity a udržitelnosti.

Ensemble metody představují významný pokrok v oblasti strojového učení a jak jsme viděli, nabízejí výkonné nástroje pro řešení klasifikačních problémů v různých doménách.